# Tarea 1 – Parte 1: Recolección y limpieza de noticias

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

| Configuración | Valor |
|---|---|
| Fecha de descarga | 18 de octubre de 2026 |
| Opción elegida | **B – El Comercio** (350–400 noticias, ~90–100 por categoría) |
| URL base | https://elcomercio.pe |
| Reglas del sitio | https://elcomercio.pe/robots.txt |
| Fuente principal (HTML) | `https://elcomercio.pe/archivo/<seccion>/` |
| Fuente de respaldo (RSS) | `https://elcomercio.pe/arcio/rss/category/<seccion>/` |

Categorías finales (fijas): `politica`, `economia`, `deportes`, `ciencia_tecnologia`.

**Plan:** 1) leer el `robots.txt`; 2) probar si el HTML responde o hay bloqueo;
3) si hay bloqueo en alguna sección, usar su RSS oficial y documentarlo.

## 1. Configuración

Buenas prácticas pedidas por el enunciado:
- `User-Agent` descriptivo (quiénes somos y para qué).
- Pausa de 1 a 2 segundos entre solicitudes.
- Máximo 1 reintento.
- Respaldo crudo de cada respuesta en `datos/raw/`.

In [1]:
import random
import re
import time
from datetime import date
from pathlib import Path
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests

BASE = "https://elcomercio.pe"
USER_AGENT = "QLAB-PythonIntermedio-Tarea1/1.0 (proyecto academico; titulares para clasificacion)"
PAUSA = (1.0, 2.0)          # segundos entre solicitudes
MAX_REINTENTOS = 1
RAW = Path("datos/raw")
RAW.mkdir(parents=True, exist_ok=True)

FECHA_DESCARGA = date.today().isoformat()
sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT})
print("Fecha de descarga:", FECHA_DESCARGA)

Fecha de descarga: 2026-10-18


## 2. Lectura del `robots.txt`

El `robots.txt` no es una herramienta de descarga: es el **reglamento** del sitio.
Lo descargamos, lo guardamos como evidencia y usamos `RobotFileParser` de la librería
estándar para preguntar, URL por URL, si nuestro `User-Agent` tiene permiso.

In [2]:
resp = sesion.get(f"{BASE}/robots.txt", timeout=20)
(RAW / "robots_elcomercio.txt").write_text(resp.text, encoding="utf-8")

# Reglas que aplican a todos los agentes (User-agent: *)
bloque_general = resp.text.split("# BLOQUEO")[0]
print(bloque_general.strip())

# DIRECTIVAS GENERALES
User-agent: *
Disallow: /catalogo/
Disallow: /buscar/
Disallow: /archivo/todas/19*
Disallow: /archivo/todas/200*


In [3]:
robots = RobotFileParser()
robots.parse(resp.text.splitlines())

# Una seccion del diario por cada categoria final
SECCIONES = {
    "politica": "politica",
    "economia": "economia",
    "deportes": "deporte-total",
    "ciencia_tecnologia": "tecnologia",
}

urls_a_revisar = []
for categoria, seccion in SECCIONES.items():
    urls_a_revisar += [
        (categoria, "HTML archivo", f"{BASE}/archivo/{seccion}/"),
        (categoria, "RSS", f"{BASE}/arcio/rss/category/{seccion}/"),
    ]
urls_a_revisar.append(("(control)", "ruta prohibida", f"{BASE}/buscar/economia/"))

permisos = pd.DataFrame(
    [(c, t, u, robots.can_fetch(USER_AGENT, u)) for c, t, u in urls_a_revisar],
    columns=["categoria", "tipo", "url", "permitido"],
)
permisos

,categoria,tipo,url,permitido
0,politica,HTML archivo,https://elcomercio.pe/archivo/politica/,True
1,politica,RSS,https://elcomercio.pe/arcio/rss/category/polit...,True
2,economia,HTML archivo,https://elcomercio.pe/archivo/economia/,True
3,economia,RSS,https://elcomercio.pe/arcio/rss/category/econo...,True
4,deportes,HTML archivo,https://elcomercio.pe/archivo/deporte-total/,True
5,deportes,RSS,https://elcomercio.pe/arcio/rss/category/depor...,True
6,ciencia_tecnologia,HTML archivo,https://elcomercio.pe/archivo/tecnologia/,True
7,ciencia_tecnologia,RSS,https://elcomercio.pe/arcio/rss/category/tecno...,True
8,(control),ruta prohibida,https://elcomercio.pe/buscar/economia/,False


La fila de control (`/buscar/`) debe salir `False`: así comprobamos que el parser
sí aplica las reglas y no responde `True` para todo.

## 3. Función de descarga responsable

`descargar()` hace una sola cosa con cuidado:
1. Si la URL ya está en `datos/raw/`, la lee del disco (no molesta al servidor otra vez).
2. Verifica el permiso en `robots.txt` antes de pedirla.
3. Hace la solicitud con pausa aleatoria de 1–2 s y **un** reintento si falla.
4. Guarda la respuesta cruda.

In [4]:
def nombre_cache(url: str) -> Path:
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", url.removeprefix(BASE)).strip("_") or "inicio"
    return RAW / f"{slug}.html"


def descargar(url: str) -> tuple[int, str]:
    """Devuelve (codigo_http, texto). Usa cache en datos/raw/ y respeta robots.txt."""
    archivo = nombre_cache(url)
    if archivo.exists():
        return 200, archivo.read_text(encoding="utf-8")
    if not robots.can_fetch(USER_AGENT, url):
        return -1, ""  # prohibido por robots.txt: ni siquiera se solicita

    codigo = 0  # 0 = sin respuesta (error de red)
    for intento in range(1 + MAX_REINTENTOS):
        time.sleep(random.uniform(*PAUSA))
        try:
            r = sesion.get(url, timeout=20)
        except requests.RequestException as e:
            print(f"  intento {intento + 1} fallo: {e}")
            continue
        codigo = r.status_code
        if codigo == 200:
            archivo.write_text(r.text, encoding="utf-8")
            return 200, r.text
        print(f"  intento {intento + 1}: HTTP {codigo}")
    return codigo, ""

## 4. Prueba de bloqueo

Un bloqueo puede verse de varias formas, no solo como un error:
- **HTTP 403 / 429 / 503**: el servidor rechaza o limita las solicitudes.
- **HTTP 200 pero sin noticias**: página de captcha, aviso de "acceso denegado"
  o contenido que se carga con JavaScript (que `requests` no ejecuta).

Por eso contamos cuántos titulares (`story-item__title`) trae realmente cada página HTML
y cuántos `<item>` trae cada RSS.

In [5]:
def contar_noticias(tipo: str, texto: str) -> int:
    if tipo == "RSS":
        return texto.count("<item>")
    return len(set(re.findall(r'class="story-item__title[^"]*"[^>]*href="([^"]+)"', texto)))


resultados = []
for categoria, tipo, url in urls_a_revisar[:-1]:   # sin la ruta de control
    codigo, texto = descargar(url)
    n = contar_noticias(tipo, texto) if codigo == 200 else 0
    if codigo == -1:
        estado = "prohibido por robots.txt"
    elif codigo != 200:
        estado = f"bloqueo / error HTTP {codigo}"
    elif n == 0:
        estado = "200 pero sin noticias (posible bloqueo)"
    else:
        estado = "OK"
    resultados.append((categoria, tipo, codigo, n, estado))

prueba = pd.DataFrame(resultados, columns=["categoria", "tipo", "http", "noticias", "estado"])
prueba

,categoria,tipo,http,noticias,estado
0,politica,HTML archivo,200,100,OK
1,politica,RSS,200,18,OK
2,economia,HTML archivo,200,100,OK
3,economia,RSS,200,16,OK
4,deportes,HTML archivo,200,100,OK
5,deportes,RSS,200,84,OK
6,ciencia_tecnologia,HTML archivo,200,100,OK
7,ciencia_tecnologia,RSS,200,13,OK


In [6]:
bloqueadas = prueba[(prueba.tipo == "HTML archivo") & (prueba.estado != "OK")]
if bloqueadas.empty:
    print("Sin bloqueo: todas las secciones responden por HTML. Fuente principal = HTML.")
    print("El RSS queda como respaldo automatico si una seccion fallara en la extraccion.")
else:
    print("Secciones con bloqueo en HTML (se usara su RSS):")
    print(bloqueadas[["categoria", "estado"]].to_string(index=False))

Sin bloqueo: todas las secciones responden por HTML. Fuente principal = HTML.
El RSS queda como respaldo automatico si una seccion fallara en la extraccion.


## 5. Extracción

Cada noticia del archivo HTML viene en un bloque `div.story-item__left` con:

| Dato | Etiqueta HTML |
|---|---|
| `titular` y `url` | `a.story-item__title` (texto y `href`) |
| `bajada` | `p.story-item__subtitle` |
| `fecha` | `time[datetime]` (hora UTC → la convertimos a fecha de Lima) |
| `seccion_original` | `a.story-item__section--top` (`href` como `/economia/mercados/`) |

Escribimos `seccion_original` como en la web, uniendo la ruta con guiones:
`/economia/mercados/` → `economia-mercados`.

Si el HTML de una sección fallara (error o 0 noticias), la función usa automáticamente
el **RSS oficial** de esa sección. Así el scraping no se cae a la primera.

In [7]:
from bs4 import BeautifulSoup

ZONA_LIMA = "America/Lima"


def ruta_a_seccion(href: str) -> str:
    """'/economia/mercados/' -> 'economia-mercados'"""
    partes = [p for p in href.split("?")[0].split("/") if p and "elcomercio.pe" not in p and p not in ("https:", "http:")]
    return "-".join(partes)


def fecha_lima(valor: str) -> str:
    return pd.to_datetime(valor, utc=True).tz_convert(ZONA_LIMA).strftime("%Y-%m-%d")


def extraer_html(texto: str) -> list[dict]:
    sopa = BeautifulSoup(texto, "lxml")
    filas = []
    for bloque in sopa.select("div.story-item__left"):
        titulo = bloque.select_one("a.story-item__title")
        if titulo is None:
            continue
        bajada = bloque.select_one("p.story-item__subtitle")
        seccion = bloque.select_one("a.story-item__section--top")
        hora = bloque.select_one("time[datetime]")
        filas.append({
            "titular": titulo.get_text(" ", strip=True),
            "bajada": bajada.get_text(" ", strip=True) if bajada else "",
            "medio": "El Comercio",
            "seccion_original": ruta_a_seccion(seccion["href"]) if seccion else "",
            "url": BASE + titulo["href"] if titulo["href"].startswith("/") else titulo["href"],
            "fecha": fecha_lima(hora["datetime"]) if hora else "",
        })
    return filas


def extraer_rss(texto: str) -> list[dict]:
    sopa = BeautifulSoup(texto, "xml")
    filas = []
    for item in sopa.find_all("item"):
        url = item.link.get_text(strip=True)
        filas.append({
            "titular": item.title.get_text(" ", strip=True),
            "bajada": BeautifulSoup(item.description.get_text(), "lxml").get_text(" ", strip=True) if item.description else "",
            "medio": "El Comercio",
            # en el RSS la seccion se deduce de la ruta de la URL, sin el slug final de la noticia
            "seccion_original": ruta_a_seccion("/".join(url.split("/")[:-2]) + "/"),
            "url": url,
            "fecha": fecha_lima(item.pubDate.get_text(strip=True)) if item.pubDate else "",
        })
    return filas

In [8]:
registros = []
fuente_usada = {}

for categoria_busqueda, seccion in SECCIONES.items():
    codigo, texto = descargar(f"{BASE}/archivo/{seccion}/")
    filas = extraer_html(texto) if codigo == 200 else []
    fuente = "HTML archivo"
    if not filas:  # plan B: RSS oficial
        print(f"{seccion}: HTML sin noticias (HTTP {codigo}), se usa RSS")
        codigo, texto = descargar(f"{BASE}/arcio/rss/category/{seccion}/")
        filas = extraer_rss(texto) if codigo == 200 else []
        fuente = "RSS"
    for f in filas:
        f["pagina_origen"] = seccion
    fuente_usada[seccion] = (fuente, len(filas))
    registros += filas

df_crudo = pd.DataFrame(registros)
display(pd.DataFrame(fuente_usada, index=["fuente", "noticias"]).T)
print("Noticias extraidas (antes de limpiar):", len(df_crudo))
df_crudo.head()

,fuente,noticias
politica,HTML archivo,100
economia,HTML archivo,100
deporte-total,HTML archivo,100
tecnologia,HTML archivo,100


Noticias extraidas (antes de limpiar): 400


,titular,bajada,medio,seccion_original,url,fecha,pagina_origen
0,El voto 31: Este es el escenario que muestra e...,Los cinco senadores del partido de Ricardo Bel...,El Comercio,politica,https://elcomercio.pe/politica/el-voto-31-este...,2026-10-18,politica
1,Jorge Nieto dice que se reunió con el Ejecutiv...,El líder del Partido del Buen Gobierno reveló ...,El Comercio,politica-gobierno,https://elcomercio.pe/politica/gobierno/jorge-...,2026-10-18,politica
2,Álvaro Vargas Llosa: “EE.UU. nos está ayudando...,"Asumirá el cargo el 16 de noviembre, pero en e...",El Comercio,politica,https://elcomercio.pe/politica/alvaro-vargas-l...,2026-10-18,politica
3,"El Senado también se la juega, una crónica de ...",Tras la agónica aprobación de facultades acota...,El Comercio,politica,https://elcomercio.pe/politica/el-senado-tambi...,2026-10-18,politica
4,Comisión de Constitución del Senado reanuda es...,El grupo de trabajo que encabeza el senador Ca...,El Comercio,politica,https://elcomercio.pe/politica/comision-de-con...,2026-10-18,politica


## 6. Limpieza y normalización

Orden de los pasos (cada uno se reporta para saber cuántas noticias se pierden y por qué):
1. Quitar espacios sobrantes en los textos.
2. Eliminar titulares vacíos.
3. Eliminar duplicados por `url` (una noticia puede aparecer en el archivo de dos secciones).
4. Mapear `seccion_original` a las 4 categorías y descartar lo que no encaja.

In [9]:
df = df_crudo.copy()
for col in ["titular", "bajada", "seccion_original"]:
    df[col] = df[col].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()

descartes = []

vacios = df[df.titular == ""]
descartes += [(u, "titular vacio") for u in vacios.url]
df = df[df.titular != ""]

duplicados = df[df.duplicated("url", keep="first")]
descartes += [(u, "url duplicada") for u in duplicados.url]
df = df.drop_duplicates("url", keep="first")

print("Titulares vacios eliminados:", len(vacios))
print("URLs duplicadas eliminadas:", len(duplicados))
print("Quedan:", len(df))

Titulares vacios eliminados: 0
URLs duplicadas eliminadas: 1
Quedan: 399


### Tabla de equivalencias sección → categoría

Solo se descargaron 4 páginas de archivo, pero **cada noticia trae su propia sección**,
que no siempre coincide con la página donde apareció. Por ejemplo, una columna sobre el
Congreso aparece en `/archivo/politica/` pero su sección propia es `opinion-columnistas`.

Algunas secciones propias no son temas sino **formatos o secciones transversales**:
`opinion` (columnas), `ecdata` (periodismo de datos), `bbc-news-mundo` (contenido BBC) y
`ruedas-tuercas` (autos). Por eso la regla tiene dos pasos:

1. **Sección propia**: si su primer tramo corresponde a una de las 4 categorías, se usa.
   (La flor bautizada en honor a Messi es `tecnologia-ciencias` → `ciencia_tecnologia`,
   aunque apareció en el archivo de deportes.)
2. **Página de archivo**: si la sección propia es transversal, se usa la categoría de la
   página donde el diario la archivó. (Los autos eléctricos aparecieron en tecnología.)

Lo que no cumple ninguna de las dos reglas se descarta; no se crea una quinta categoría.

In [10]:
MAPEO_PREFIJO = {
    "politica": "politica",
    "elecciones": "politica",
    "economia": "economia",
    "deporte-total": "deportes",
    "tecnologia": "ciencia_tecnologia",
    "eldominical-ciencia": "ciencia_tecnologia",
}
# Secciones que no indican un tema: se usa la pagina de archivo donde aparecieron
TRANSVERSALES = ["opinion", "ecdata", "bbc-news-mundo", "ruedas-tuercas"]
CATEGORIA_DE_PAGINA = {seccion: categoria for categoria, seccion in SECCIONES.items()}


def es_prefijo(seccion: str, prefijo: str) -> bool:
    return seccion == prefijo or seccion.startswith(prefijo + "-")


def mapear(seccion: str, pagina: str) -> tuple:
    """Devuelve (categoria, regla aplicada). categoria=None si se descarta."""
    for prefijo, categoria in MAPEO_PREFIJO.items():
        if es_prefijo(seccion, prefijo):
            return categoria, "seccion propia"
    if any(es_prefijo(seccion, t) for t in TRANSVERSALES):
        return CATEGORIA_DE_PAGINA[pagina], f"pagina de archivo ({pagina})"
    return None, "sin categoria"


df[["etiqueta", "regla"]] = [mapear(s, p) for s, p in zip(df.seccion_original, df.pagina_origen)]

tabla_mapeo = (
    df.groupby(["seccion_original", "regla"], dropna=False)
      .agg(noticias=("url", "size"), etiqueta=("etiqueta", "first"))
      .reset_index()
      .fillna({"etiqueta": "DESCARTADA"})
      .sort_values(["etiqueta", "noticias"], ascending=[True, False])
      [["seccion_original", "etiqueta", "regla", "noticias"]]
)
pd.set_option("display.max_rows", 80)
tabla_mapeo.reset_index(drop=True)

,seccion_original,etiqueta,regla,noticias
0,tecnologia-actualidad,ciencia_tecnologia,seccion propia,37
1,tecnologia-inteligencia-artificial,ciencia_tecnologia,seccion propia,19
2,tecnologia-ciencias,ciencia_tecnologia,seccion propia,10
3,tecnologia-ciberseguridad,ciencia_tecnologia,seccion propia,7
4,tecnologia-ecologia,ciencia_tecnologia,seccion propia,6
5,tecnologia-videojuegos,ciencia_tecnologia,seccion propia,5
6,tecnologia-moviles,ciencia_tecnologia,seccion propia,4
7,tecnologia-gadgets,ciencia_tecnologia,seccion propia,3
8,bbc-news-mundo,ciencia_tecnologia,pagina de archivo (tecnologia),2
9,ruedas-tuercas-automotriz,ciencia_tecnologia,pagina de archivo (tecnologia),2


In [11]:
# Las noticias que cambiaron de categoria por la regla 2, para revisarlas a ojo
df[df.regla.str.startswith("pagina")][["titular", "seccion_original", "etiqueta"]]

,titular,seccion_original,etiqueta
8,Cómo entender el Congreso peruano,opinion-columnistas,politica
9,Empiecen a gobernar,opinion-columnistas,politica
15,La dispersión del voto creció en las últimas e...,ecdata,politica
26,“Amarrando la chiva”,opinion-columnistas,politica
47,"Si no es por el fondo, al menos por las formas",opinion-columnistas,politica
56,Elecciones municipales 2026: Casi mil candidat...,ecdata,politica
67,Una conferencia legislativa,opinion-columnistas,politica
85,"¡A serenarse, señora Fujimori!",opinion-columnistas,politica
111,Fenómeno El Niño: Presupuesto para prevención ...,ecdata,economia
275,Guaylupo y Escuza ganaron en “Contender Series...,ecdata,deportes


In [12]:
fuera = df[df.etiqueta.isna()]
descartes += [(u, f"categoria no admitida ({s})") for u, s in zip(fuera.url, fuera.seccion_original)]
df = df[df.etiqueta.notna()].drop(columns="regla").reset_index(drop=True)

reporte_descartes = pd.DataFrame(descartes, columns=["url", "motivo"])
print("Total descartadas:", len(reporte_descartes), "| Noticias finales:", len(df))
reporte_descartes

Total descartadas: 1 | Noticias finales: 399


,url,motivo
0,https://elcomercio.pe/tecnologia/ciencias/salv...,url duplicada


In [13]:
df.etiqueta.value_counts().rename("noticias").to_frame()

,noticias
etiqueta,
politica,100
economia,100
ciencia_tecnologia,100
deportes,99
